# 2 · Azure OpenAI tool calling

The model requests a named function with JSON arguments; Python validates and executes it, then returns a tool message using the matching call ID. Configure the root `.env`. Running Azure cells may incur charges. No live Azure outputs are stored.

In [ ]:
from pathlib import Path
import sys
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "assistant_core.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from inside the repository.")
sys.path.insert(0, str(ROOT))

In [ ]:
import json
from assistant_core import Settings, WEATHER_TOOLS, ORDER_TOOLS
from assistant_core import get_demo_weather, run_tool_chat, answer_order_query
settings = Settings.load()
print(json.dumps(ORDER_TOOLS, indent=2))

## A local demonstration function
This weather helper returns fixed sample data. Unit conversion is consistent: 72°F is approximately 22.2°C. It does not fetch live observations.

In [ ]:
print(get_demo_weather("Paro", unit="fahrenheit"))
print(get_demo_weather("Paro", unit="celsius"))

## Let the model request the function
`tool_events` shows the executed calls and their results. Inspect `run_tool_chat` in `assistant_core.py` to see the complete request/dispatch/tool-result loop.

In [ ]:
with settings.client() as client:
    weather = run_tool_chat("Show me the demo weather for Paro in celsius.",
        client=client, deployment=settings.deployment, tools=WEATHER_TOOLS,
        handlers={"get_demo_weather": get_demo_weather})
print(weather.answer)
print(json.dumps(weather.tool_events, indent=2))

## A direct answer may need no tool
The code checks for `tool_calls` before extracting arguments. It does not assume every response requests a function.

In [ ]:
with settings.client() as client:
    direct = run_tool_chat("Explain what a REST API is in one sentence.",
        client=client, deployment=settings.deployment, tools=WEATHER_TOOLS,
        handlers={"get_demo_weather": get_demo_weather})
print(direct.answer)
print("Executed tools:", len(direct.tool_events))

## Order lookup through a REST API
Start the FastAPI server before running the next cells. Order identifiers are strings so formatting is not lost. The server validates the requested order number.

In [ ]:
order = answer_order_query("When is demo order 123456789 expected to arrive?", settings=settings)
print(order.answer)
print(json.dumps(order.tool_events, indent=2))

In [ ]:
missing = answer_order_query("Look up order 999999999.", settings=settings)
print(missing.answer)
print(json.dumps(missing.tool_events, indent=2))

## Review the protocol
The loop sends the assistant's `tool_calls`, executes only registered functions, and appends a `role="tool"` message for every call with its `tool_call_id`. Tool failures become structured results. The model then produces a reply. The default limit is two tool rounds, with up to four calls per round. The final allowed request disables tools. The application validates arguments because a schema alone does not guarantee valid output.

In [ ]:
shipping = answer_order_query(
    "What percentage of the total is shipping for demo order 123456789?", settings=settings)
print(shipping.answer)

The API computes `ShippingPercentageOfTotal` as `shipping / (item value + shipping) × 100`. Inspect the tool result rather than treating model arithmetic as verified. The Streamlit app exposes order tools only; the weather tool is used only in this notebook.